## OpenSource Routing Models:

[vela 2.0](https://vllm-sr.ai/blog/vela-2-0-open-foundation-routing-models/)

In [1]:
!pip install torch "transformers>=5.17" safetensors tokenizers numpy

In [ ]:
from transformers import AutoModel

m = AutoModel.from_pretrained("vllm-sr/Vela-2.0-0.3B", trust_remote_code=True)
result = m.system_one(
    state={"request": "Email Tom Baker at tom.baker@example.com."},
    questions={
        "route": {"type": "choice", "instructions": "What does this request ask for?",
                  "criteria": {"email": "send an email", "other": "another task"},
                  "over": "request"},
        "pii": {"type": "span", "instructions": "Which spans are personal information?",
                "criteria": m.vela2_engine.cal["pii_schema"]["labels"], "over": "request"}})
print(result["answers"])
print(result["spans"])

Loading weights:   0%|          | 0/152 [00:00<?, ?it/s]

{'route': {'type': 'choice', 'choice': 'email', 'confidence': 0.9809066117465493, 'probabilities': {'email': 0.9904533058732746, 'other': 0.009546694126725354}}, 'pii': {'type': 'noul', 'noul': 0.9999906391084431}}
{'pii': [{'label': 'PERSON', 'start': 6, 'end': 15, 'text': 'Tom Baker', 'probability': 0.9999293088912964}, {'label': 'EMAIL_ADDRESS', 'start': 19, 'end': 40, 'text': 'tom.baker@example.com', 'probability': 0.9998665452003479}]}


In [ ]:
result

{'model': 'vllm-sr/Vela-2.0-0.3B',
 'answers': {'route': {'type': 'choice',
   'choice': 'email',
   'confidence': 0.9809066117465493,
   'probabilities': {'email': 0.9904533058732746,
    'other': 0.009546694126725354}},
  'pii': {'type': 'noul', 'noul': 0.9999906391084431}},
 'usage': {'input_tokens': 228, 'output_tokens': 0},
 'spans': {'pii': [{'label': 'PERSON',
    'start': 6,
    'end': 15,
    'text': 'Tom Baker',
    'probability': 0.9999293088912964},
   {'label': 'EMAIL_ADDRESS',
    'start': 19,
    'end': 40,
    'text': 'tom.baker@example.com',
    'probability': 0.9998665452003479}]},
 'thresholds': {'pii': 0.75}}

In [ ]:
from transformers import AutoModel

# m = AutoModel.from_pretrained("vllm-sr/Vela-2.0-9B", trust_remote_code=True)
PII_LABELS = m.vela2_engine.cal["pii_schema"]["labels"]   # the 17 trained PII types


In [ ]:
result = m.system_one(
    state={"request": "Hi, I'm Tom Baker (tom.baker@example.com). What is the maximum daily dose of paracetamol for an adult?",
           "source": "For adults, the maximum dose of paracetamol is 4 grams in 24 hours, taken as 500 mg to 1 g every 4 to 6 hours.",
           "answer": "Adults can take up to 6 grams of paracetamol in 24 hours, in doses of 500 mg to 1 g every 4 to 6 hours."},
    questions={
        "pii": {"type": "span", "instructions": "Which spans are personal information?",
                "criteria": PII_LABELS, "over": "request"},
        "halu": {"type": "span", "instructions": "Which spans of the answer are not supported by the context?",
                 "criteria": {"unsupported": "a claim not supported by the context"}},
        "domain": {"type": "choice", "instructions": "Which subject area is this request about?", "over": "request",
                   "criteria": {"health": "medicine, clinical practice, nutrition, ageing or sexual health",
                                "math": "arithmetic, algebra, geometry, statistics or other mathematics",
                                "other": "a subject that fits none of the listed areas"}}})

In [ ]:
result

{'model': 'vllm-sr/Vela-2.0-0.3B',
 'answers': {'pii': {'type': 'noul', 'noul': 0.9998887706413214},
  'halu': {'type': 'noul', 'noul': 0.9844038656175801},
  'domain': {'type': 'choice',
   'choice': 'health',
   'confidence': 0.5101261125724448,
   'probabilities': {'health': 0.7272232247432955,
    'math': 0.21709711217085073,
    'other': 0.05567966308585381}}},
 'usage': {'input_tokens': 471, 'output_tokens': 0},
 'spans': {'pii': [{'label': 'PERSON',
    'start': 8,
    'end': 17,
    'text': 'Tom Baker',
    'probability': 0.9998863935470581},
   {'label': 'EMAIL_ADDRESS',
    'start': 19,
    'end': 40,
    'text': 'tom.baker@example.com',
    'probability': 0.9833777546882629}],
  'halu': [{'label': 'unsupported',
    'start': 16,
    'end': 29,
    'text': 'up to 6 grams',
    'probability': 0.8949968814849854}]},
 'thresholds': {'pii': 0.46046470810296286, 'halu': 0.55}}